### Import Dependencies

In [1]:
from google import genai
from google.genai import types
import os
import json


from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client



In [2]:
### Download all the data from qdrant

qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [6]:
all_points[0][0].payload

{'preprocessed_description': 'MANDDDWU 80x100 Monocular-Telescope Low Night Vision Monoculars High Definition for Adults High Powered with Smartphone Adapter Telescope Hunting Wildlife Bird Watching Travel Camping Hiking ',
 'image_url': 'https://m.media-amazon.com/images/I/51h4NUuSazL._AC_.jpg',
 'rating_number': 446,
 'price': 29.99,
 'average_rating': 3.8,
 'parent_asin': 'B09QQ2J8XN'}

In [7]:
all_contexts = [{"id":point.payload["parent_asin"], "text":point.payload["preprocessed_description"]} for point in all_points[0]]

In [8]:
all_contexts

[{'id': 'B09QQ2J8XN',
  'text': 'MANDDDWU 80x100 Monocular-Telescope Low Night Vision Monoculars High Definition for Adults High Powered with Smartphone Adapter Telescope Hunting Wildlife Bird Watching Travel Camping Hiking '},
 {'id': 'B08DLWKXGL',
  'text': 'Car Windshield Dashboard Tablet Holder for 5.5-12.9" Tablet Smartphone, EXSHOW Long Arm Suction Cup Car Window Tablet Cell Phone Mount for iPad Pro Air Mini, Samsung Galaxy Tab, iPhone, Xiaomi etc '},
 {'id': 'B0C6MJL8D8',
  'text': 'SOOMFON Bluetooth 5.0 FM Transmitter for Car, Stronger Microphone & Bass Sound Bluetooth Radio Transmitter Car Adapter, Support 20W PD+QC3.0, 7 Colors LED Backlit, Wireless Call, AUX Output & U Disk 【Hi-Fi & Deep Bass Sound】SOOMFON Bluetooth car adapter with powerful bass-driven tech, you can enjoy 100% more bass than ordinary Bluetooth FM transmitter via the Long press of the "Light switch"button, display show b0 means normal. b1 means bass boost. High speed noise cancellation and BASS sound selecti

In [9]:
### Render a prompt to generate synthetic eval dataset

output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "reasoning":{
            "type": "string",
            "description": "Reasoning why the question could be answered with the chunks.",
        },
        "properties": {
            "type": "string",
            "description": "Suggested question",
        },
        "chunk_ids":{
            "type": "array",
            "description": "List of chunk ids that are relevant to the question",
        },
        "answer_example":{
            "type": "string",
            "description": "Suggested answer to the question grounded in the context",
        }
    }
}

SYSTEM_PROMPT = f"""

I am building a RAG application, I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products that we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use a single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use the word "chunks" in the suggested questions, refer to the chunks as products.

<output JSON schema>
{json.dumps(output_schema, indent=2)}
</output JSON schema>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of shunks, each list element is a dictionary with id and text:
{json.dumps(all_contexts, indent=2)}
"""

In [10]:
print(USER_PROMPT)


Here is the list of shunks, each list element is a dictionary with id and text:
[
  {
    "id": "B09QQ2J8XN",
    "text": "MANDDDWU 80x100 Monocular-Telescope Low Night Vision Monoculars High Definition for Adults High Powered with Smartphone Adapter Telescope Hunting Wildlife Bird Watching Travel Camping Hiking "
  },
  {
    "id": "B08DLWKXGL",
    "text": "Car Windshield Dashboard Tablet Holder for 5.5-12.9\" Tablet Smartphone, EXSHOW Long Arm Suction Cup Car Window Tablet Cell Phone Mount for iPad Pro Air Mini, Samsung Galaxy Tab, iPhone, Xiaomi etc "
  },
  {
    "id": "B0C6MJL8D8",
    "text": "SOOMFON Bluetooth 5.0 FM Transmitter for Car, Stronger Microphone & Bass Sound Bluetooth Radio Transmitter Car Adapter, Support 20W PD+QC3.0, 7 Colors LED Backlit, Wireless Call, AUX Output & U Disk \u3010Hi-Fi & Deep Bass Sound\u3011SOOMFON Bluetooth car adapter with powerful bass-driven tech, you can enjoy 100% more bass than ordinary Bluetooth FM transmitter via the Long press of the \

In [11]:
print(SYSTEM_PROMPT)



I am building a RAG application, I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products that we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use a single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use the word "chunks" in the sugges

In [15]:
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = gemini_client.models.generate_content(
    contents=[SYSTEM_PROMPT, USER_PROMPT],
    model="gemini-3.5-flash",
)

print(response.text)

```json
[
  {
    "properties": "I'm looking for an 80x100 monocular telescope for bird watching. Do you have options with a smartphone adapter?",
    "reasoning": "Both products represent 80x100 monocular telescopes designed for adults, supporting bird watching, and equipped with a smartphone adapter.",
    "chunk_ids": [
      "B09QQ2J8XN",
      "B09YHBXZC8"
    ],
    "answer_example": "Yes, we have two 80x100 Monocular-Telescopes in stock (IDs: B09QQ2J8XN and B09YHBXZC8) that are perfect for bird watching, wildlife viewing, and hiking. Both models are designed for adults and come with a smartphone adapter."
  },
  {
    "properties": "Do you sell any 3-packs of 6ft nylon braided iPhone charger cables? What options are available?",
    "reasoning": "Both products describe 3-packs of 6-foot MFi-certified nylon braided iPhone charging cables supporting 2.4A fast charging and 480 Mbps data transfer.",
    "chunk_ids": [
      "B0C93Q4CPK",
      "B0BLBJFKNT"
    ],
    "answer_example

In [16]:
print(response.usage_metadata.total_token_count)

32762


In [17]:
json_output = response.text

In [18]:
json_output = json_output.replace("```json", "").replace("```", "")

In [19]:
json_output = json.loads(json_output)

In [20]:
single_chunk_counter = 0
multiple_chunk_counter = 0
unanswerable_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        unanswerable_counter += 1

In [21]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Unanswerable questions: {unanswerable_counter}")

Single chunk questions: 15
Multiple chunk questions: 10
Unanswerable questions: 5


In [25]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0C93Q4CPK")
            )
        ]
    )
)[0]

In [26]:
points[0].payload

{'preprocessed_description': "iPhone Charger Cable 3Pack 6FT, [Apple MFi Certified] Lightning Cable Cord Nylon Braided iPhone Charger Fast Charging Compatible with iPhone 14 13 12 11 Pro Max XR XS X 8 7 6 Plus SE/iPad 【Fast Charge & Data Transfer】-iPhone charger cable is made of high-purity copper core and intelligent chip, with overcharge protection, stable current protection, automatic switching, and battery protection, and can support up to 2.4A fast charging and 480 Mbps transfer speed. 【Apple Mfi Certified】-Certified by MFI and original 8 Pin connector with a fast Charging end, ensured safe charging for your devices. Enjoy fast charge and data transfer. 【Super Durability & Flexibility】- iphone charger cable with premium double shade braided nylon body to ensure it's super durability and strength.The polished aluminum alloy case not only fits your iPhone perfectly, but also prevents physical compression. 【Compatible Models】-Lightning cable compatible with iPhone 13/12/ 11 / 11 Pro 

In [27]:
def get_product_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]
    return points[0].payload["preprocessed_description"]

In [28]:
get_product_description("B0C93Q4CPK")

"iPhone Charger Cable 3Pack 6FT, [Apple MFi Certified] Lightning Cable Cord Nylon Braided iPhone Charger Fast Charging Compatible with iPhone 14 13 12 11 Pro Max XR XS X 8 7 6 Plus SE/iPad 【Fast Charge & Data Transfer】-iPhone charger cable is made of high-purity copper core and intelligent chip, with overcharge protection, stable current protection, automatic switching, and battery protection, and can support up to 2.4A fast charging and 480 Mbps transfer speed. 【Apple Mfi Certified】-Certified by MFI and original 8 Pin connector with a fast Charging end, ensured safe charging for your devices. Enjoy fast charge and data transfer. 【Super Durability & Flexibility】- iphone charger cable with premium double shade braided nylon body to ensure it's super durability and strength.The polished aluminum alloy case not only fits your iPhone perfectly, but also prevents physical compression. 【Compatible Models】-Lightning cable compatible with iPhone 13/12/ 11 / 11 Pro / 11 Pro Max / XS / XS Max / 

### Create eval dataset in Langsmith

In [29]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [31]:
ls_client = Client()

In [32]:
dataset_name = "RAG-Eval-Dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic eval dataset for RAG system",
)

In [38]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["properties"],
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions":[get_product_description(id) for id in item["chunk_ids"]]
        }
    )